# Module 22: ML Project Architecture — Solutions

Complete solutions to all exercises.

In [ ]:
import yaml
import os
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any
from contextlib import contextmanager
import datetime

### Solution 1: Project Structure Design

In [ ]:
print("""
credit-risk/
  src/
    ingestion/
      __init__.py
      csv_loader.py        # Load from CSV files
      api_loader.py         # Load from REST APIs
      db_loader.py          # Load from PostgreSQL
      schemas.py            # Data schema definitions
    features/
      __init__.py
      credit_features.py    # Credit-specific features
      behavioral_features.py# Transaction behavioral features
      transformers.py       # Sklearn-compatible transformers
      validators.py          # Feature quality validation
    training/
      __init__.py
      trainer.py            # Model training orchestration
      tuner.py              # Hyperparameter tuning
      evaluator.py          # Model evaluation & comparison
      selectors.py          # Best model selection logic
    serving/
      __init__.py
      predictor.py          # Inference with trained model
      api.py                # FastAPI endpoint
      ab_test.py            # A/B testing router
    monitoring/
      __init__.py
      drift_detector.py     # Data/prediction drift
      performance_tracker.py # Live performance metrics
    common/
      __init__.py
      config.py             # Configuration management
      logging.py            # Structured logging setup
      metrics.py            # Custom metric definitions
  config/
    default.yaml
    dev.yaml
    staging.yaml
    production.yaml
  tests/
    test_ingestion/
    test_features/
    test_training/
    test_serving/
  models/                    # Model artifacts
  data/                      # Data files (gitignored)
  notebooks/                 # Exploration
  orchestration/             # Airflow/Prefect DAGs
  docker/
    Dockerfile
    docker-compose.yaml
  docs/
    architecture.md
    api_spec.md
""")

### Solution 2: pydantic-settings Configuration

In [ ]:
from pydantic import Field, field_validator, ValidationError


class MLSettings(BaseSettings):
    db_url: str = Field(
        default="postgresql://localhost:5432/ml_db",
        description="Database connection URL"
    )
    model_path: str = Field(default="models/latest.pkl")
    api_key: str = Field(default="", description="External API key")
    max_depth: int = Field(default=10, ge=1, le=100)
    n_estimators: int = Field(default=100, ge=10, le=10000)
    learning_rate: float = Field(default=0.01, gt=0.0, le=1.0)
    batch_size: int = Field(default=64, ge=1, le=1024)

    class Config:
        env_prefix = "ML_"
        env_file = ".env"

    @field_validator("db_url")
    def validate_db_url(cls, v):
        if not v.startswith("postgresql://") and not v.startswith("sqlite://"):
            raise ValueError("db_url must start with postgresql:// or sqlite://")
        return v


print("MLSettings defined with pydantic-settings")
print("Environment variables: ML_DB_URL, ML_MODEL_PATH, ML_API_KEY, etc.")

### Solution 3: Experiment Tracker Context Manager

In [ ]:
import hashlib


class ExperimentRunLocal:
    """Fallback experiment tracker when MLflow is unavailable."""
    def __init__(self, run_name=None):
        self.run_name = run_name or f"run_{hashlib.md5(str(datetime.datetime.now()).encode()).hexdigest()[:8]}"
        self.params = {}
        self.metrics = {}

    def log_params(self, params_dict):
        self.params.update(params_dict)

    def log_metrics(self, metrics_dict):
        self.metrics.update(metrics_dict)

    def log_model(self, model, path):
        print(f"[TRACKER] Model saved to {path}")

    def get_summary(self):
        return {"run_name": self.run_name, "params": self.params, "metrics": self.metrics}


@contextmanager
def ExperimentRun(experiment_name, tracking_uri=None, use_mlflow=False):
    run_name = f"{experiment_name}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    print(f"[EXPERIMENT] Starting run: {run_name}")
    if use_mlflow:
        try:
            import mlflow
            mlflow.set_tracking_uri(tracking_uri or "http://localhost:5000")
            mlflow.set_experiment(experiment_name)
            with mlflow.start_run(run_name=run_name) as run:
                yield run
        except Exception as e:
            print(f"[WARNING] MLflow failed ({e}), using local fallback")
            yield ExperimentRunLocal(run_name)
    else:
        yield ExperimentRunLocal(run_name)
    print(f"[EXPERIMENT] Completed run: {run_name}")


# Test
with ExperimentRun("credit-risk-v2") as run:
    run.log_params({"model": "xgboost", "lr": 0.01})
    run.log_metrics({"auc": 0.89, "precision": 0.85})
    print(f"Run params: {run.params}")

print("Solution 3 PASSED")

### Solution 4: Data Versioning Workflow

In [ ]:
class DataCatalog:
    """Simple data catalog for tracking dataset versions."""

    def __init__(self, catalog_path="data_catalog.json"):
        self.catalog_path = catalog_path
        self.catalog = {}
        if os.path.exists(catalog_path):
            with open(catalog_path) as f:
                self.catalog = json.load(f)

    def register_version(self, dataset_name, version, path, description=""):
        if dataset_name not in self.catalog:
            self.catalog[dataset_name] = {}
        self.catalog[dataset_name][version] = {
            "path": path,
            "description": description,
            "registered_at": datetime.datetime.now().isoformat(),
        }
        self._save()
        print(f"Registered {dataset_name} v{version} -> {path}")

    def load_version(self, dataset_name, version="latest"):
        if dataset_name not in self.catalog:
            raise KeyError(f"Dataset '{dataset_name}' not found")
        versions = self.catalog[dataset_name]
        if version == "latest":
            version = max(versions.keys())
        if version not in versions:
            raise KeyError(f"Version '{version}' not found for {dataset_name}")
        return versions[version]["path"]

    def list_versions(self, dataset_name):
        return list(self.catalog.get(dataset_name, {}).keys())

    def _save(self):
        with open(self.catalog_path, "w") as f:
            json.dump(self.catalog, f, indent=2)


catalog = DataCatalog("/tmp/test_catalog.json")
catalog.register_version("customers", "v1", "data/raw/customers_202401.csv", "Original extract")
catalog.register_version("customers", "v2", "data/raw/customers_202402.csv", "Added churn flag")
print(f"Latest customer data: {catalog.load_version('customers')}")
print(f"All versions: {catalog.list_versions('customers')}")

### Solution 5: Pipeline Orchestration DAG

In [ ]:
print("""
Prefect Flow: credit_risk_pipeline
""")


def task(func):
    """Simple task decorator for demonstration."""
    def wrapper(*args, **kwargs):
        print(f"  Running: {func.__name__}")
        return func(*args, **kwargs)
    return wrapper


@task
def extract_from_postgres():
    print("    Extracting data from PostgreSQL (daily at 2 AM)")
    return {"status": "success", "rows": 50000}

@task
def validate_quality(data):
    missing_pct = 2.0  # simulated
    if missing_pct > 5.0:
        raise ValueError(f"Data quality check failed: {missing_pct}% missing")
    print(f"    Data quality OK: {missing_pct}% missing")
    return data

@task
def compute_features(data):
    print("    Computing 50+ features...")
    return {"features": [f"feat_{i}" for i in range(50)], "samples": data["rows"]}

@task
def train_models(features):
    models = ["LR", "RF", "XGBoost"]
    results = {}
    for m in models:
        print(f"    Training {m} with 5-fold CV...")
        results[m] = {"auc": 0.80 + hash(m) % 20 * 0.01}
    return results

@task
def select_best(results):
    best = max(results, key=lambda k: results[k]["auc"])
    print(f"    Best model: {best} (AUC={results[best]['auc']:.3f})")
    return best, results[best]

@task
def register_model(model_info):
    print(f"    Registering {model_info[0]} in MLflow Registry")

@task
def deploy_to_staging(model_info, current_best_auc=0.85):
    model_name, metrics = model_info
    improvement = metrics["auc"] - current_best_auc
    if improvement > 0.01:
        print(f"    Deploying to staging (AUC improvement: {improvement:.3f})")
    else:
        print(f"    Skipping deployment (improvement {improvement:.3f} < 0.01)")

@task
def send_notification(success=True):
    status = "SUCCESS" if success else "FAILURE"
    print(f"    Notification: Pipeline {status}")


def credit_risk_pipeline():
    print("\nExecuting credit_risk_pipeline:")
    data = extract_from_postgres()
    validated = validate_quality(data)
    features = compute_features(validated)
    results = train_models(features)
    best = select_best(results)
    register_model(best)
    deploy_to_staging(best)
    send_notification(True)


credit_risk_pipeline()

### Solution 6: Hierarchical Configuration

In [ ]:
class HierarchicalConfig:
    """Load config with hierarchy: defaults < env-specific < env vars."""

    def __init__(self, config_dir="config"):
        self.config_dir = Path(config_dir)
        self._config: Dict[str, Any] = {}
        self._load()

    def _load(self):
        # 1. Load defaults
        default_path = self.config_dir / "default.yaml"
        if default_path.exists():
            with open(default_path) as f:
                self._config.update(yaml.safe_load(f) or {})

        # 2. Load environment-specific overrides
        env = os.getenv("APP_ENV", "dev")
        env_path = self.config_dir / f"{env}.yaml"
        if env_path.exists():
            with open(env_path) as f:
                env_config = yaml.safe_load(f) or {}
            self._deep_merge(self._config, env_config)

        # 3. Environment variable overrides (ML_ prefix)
        for key, value in os.environ.items():
            if key.startswith("ML_"):
                config_key = key[3:].lower().replace("__", ".")
                self._set_nested(self._config, config_key.split("."), value)

    def get(self, key, default=None):
        keys = key.split(".")
        value = self._config
        for k in keys:
            if isinstance(value, dict):
                value = value.get(k)
            else:
                return default
        return value if value is not None else default

    def _deep_merge(self, base, override):
        for key, value in override.items():
            if key in base and isinstance(base[key], dict) and isinstance(value, dict):
                self._deep_merge(base[key], value)
            else:
                base[key] = value

    def _set_nested(self, d, keys, value):
        for key in keys[:-1]:
            d = d.setdefault(key, {})
        d[keys[-1]] = value


# Demo
os.environ["APP_ENV"] = "staging"
os.environ["ML_MODEL__LEARNING_RATE"] = "0.05"
print("HierarchicalConfig class implemented")
print("Merge order: default.yaml -> staging.yaml -> env vars")

### Solution 7: Enhanced Model Registry

In [ ]:
@dataclass
class ModelEntry:
    name: str
    version: int
    path: str
    metrics: Dict[str, float]
    stage: str = "None"
    dataset_version: Optional[str] = None
    experiment_run_id: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())


class ModelRegistryV2:
    def __init__(self, promotion_threshold=0.01):
        self._models: Dict[str, List[ModelEntry]] = {}
        self.promotion_threshold = promotion_threshold

    def register(self, name, path, metrics, dataset_version=None, run_id=None):
        if name not in self._models:
            self._models[name] = []
        version = len(self._models[name]) + 1
        entry = ModelEntry(
            name=name,
            version=version,
            path=path,
            metrics=metrics,
            dataset_version=dataset_version,
            experiment_run_id=run_id,
        )
        self._models[name].append(entry)
        print(f"Registered {name} v{version}")
        return entry

    def auto_promote(self, name, new_version):
        prod = self.get_production(name)
        new_model = self.get_version(name, new_version)
        if prod is None:
            self.set_stage(name, new_version, "Production")
            print(f"First model promoted to Production")
            return True
        primary_metric = "auc"
        improvement = new_model.metrics.get(primary_metric, 0) - prod.metrics.get(primary_metric, 0)
        if improvement >= self.promotion_threshold:
            self.set_stage(name, prod.version, "Archived")
            self.set_stage(name, new_version, "Production")
            print(f"Promoted v{new_version} (improvement={improvement:.4f})")
            return True
        print(f"No promotion: improvement={improvement:.4f} < threshold={self.promotion_threshold}")
        return False

    def rollback(self, name):
        models = self._models.get(name, [])
        prod_idx = None
        for i, m in enumerate(models):
            if m.stage == "Production":
                prod_idx = i
                break
        if prod_idx is None or prod_idx == 0:
            return False
        models[prod_idx].stage = "Archived"
        models[prod_idx - 1].stage = "Production"
        print(f"Rolled back to v{models[prod_idx - 1].version}")
        return True

    def get_production(self, name):
        for m in self._models.get(name, []):
            if m.stage == "Production":
                return m
        return None

    def get_version(self, name, version):
        for m in self._models.get(name, []):
            if m.version == version:
                return m
        return None

    def set_stage(self, name, version, stage):
        m = self.get_version(name, version)
        if m:
            m.stage = stage


registry = ModelRegistryV2(promotion_threshold=0.01)
m1 = registry.register("churn", "v1.pkl", {"auc": 0.85}, "v1")
m2 = registry.register("churn", "v2.pkl", {"auc": 0.87}, "v2")
registry.auto_promote("churn", 1)
registry.auto_promote("churn", 2)
print(f"Production: {registry.get_production('churn').version}")
registry.rollback("churn")
print(f"After rollback: {registry.get_production('churn').version}")

### Solution 9: MLOps Maturity Assessment

In [ ]:
print("Scenario 1: Startup (Level 0)")
print("  Current: Colab notebooks, no version control, no tracking")
print("  Top 3 improvements:")
print("    1. Git + GitHub for code versioning")
print("    2. Organized project structure (cookiecutter DS)")
print("    3. Basic experiment tracking (MLflow local)")
print()
print("Scenario 2: Mid-size Company (Level 1)")
print("  Current: Git, CI/CD for web app, manual model training")
print("  Top 3 improvements:")
print("    1. Experiment tracking for all model runs")
print("    2. Data versioning with DVC")
print("    3. Automated model training pipeline")
print()
print("Scenario 3: Enterprise (Level 2)")
print("  Current: Experiment tracking, DVC, model registry, manual deploy")
print("  Top 3 improvements:")
print("    1. CI/CD for model deployment")
print("    2. A/B testing infrastructure")
print("    3. Automated retraining triggers")
print()
print("Scenario 4: ML-first Company (Level 3)")
print("  Current: Automated retraining, A/B testing")
print("  Top 3 improvements:")
print("    1. Prediction/data drift monitoring")
print("    2. Self-healing pipelines (auto-rollback on drift)")
print("    3. Automated model explainability reports")